In [ ]:
import pandas as pd
products_df = pd.read_csv('../csv-files/olist_products_dataset.csv')
cat_ref_df = pd.read_csv('../csv-files/product_category_name_translation.csv')
products_df.shape

(32951, 9)

In [39]:
products_df.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [40]:
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [41]:
prod_id_unq = products_df['product_id'].nunique()
prod_unq = products_df['product_category_name'].nunique()

print("unique product_id:", prod_id_unq)
print("unique product_category_name:", prod_unq)

unique product_id: 32951
unique product_category_name: 73


In [42]:
cat_ref_df.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [43]:
category_ref = cat_ref_df['product_category_name'].unique()
# print(category_ref) 71 unique category
remaining_errors = products_df[~
    products_df['product_category_name'].isin(category_ref)
]
print("has no match on our category_ref", len(remaining_errors))
print(remaining_errors['product_category_name'].value_counts(dropna=False))


has no match on our category_ref 623
product_category_name
NaN                                              610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


In [ ]:
# checking if there is a way we can populate this null values 
null_id = products_df[products_df['product_category_name'].isna()].sort_values('product_weight_g')
null_id.head()


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1285,a9d8f831888820cd641dcd5ca9fea4e8,NaN,NaN,NaN,NaN,50.0,16.0,2.0,11.0
20000,4941b2c06706332ebfd9072d7b1fb23c,NaN,NaN,NaN,NaN,50.0,32.0,23.0,33.0
3044,0502d1a36be75bd36b452f31c6ed264a,NaN,NaN,NaN,NaN,100.0,16.0,7.0,11.0
30830,4914f8796af2ecd359fd8f44b9b92339,NaN,NaN,NaN,NaN,100.0,16.0,11.0,12.0
18518,c2fffbd2b50c3f612755de49109e6d97,NaN,NaN,NaN,NaN,100.0,16.0,2.0,11.0


In [ ]:
# Drops rows only if name AND description AND photo AND quantity are ALL null because we have no way of populating them 
old_rows = len(products_df.copy())
print("original rows:", old_rows)
columns_to_check = ['product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty']
products_df = products_df.dropna(subset=columns_to_check, how='all')
print("new row number:", len(products_df))



original rows: 32951
new row number: 32341


In [46]:
cat_ref_df['eng_cat_len'] = cat_ref_df['product_category_name_english'].str.len()
cat_ref_df.head()

,product_category_name,product_category_name_english,eng_cat_len
0,beleza_saude,health_beauty,13
1,informatica_acessorios,computers_accessories,21
2,automotivo,auto,4
3,cama_mesa_banho,bed_bath_table,14
4,moveis_decoracao,furniture_decor,15


In [47]:
# 1. Merge tables
translated_df = products_df.merge(
    cat_ref_df[['product_category_name', 'product_category_name_english']], 
    on='product_category_name', 
    how='left'
)

# 2. Patch known Olist dataset omissions
missing_map = {
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_and_food_preparers',
    'pc_gamer': 'pc_gamer'
}
translated_df['product_category_name_english'] = translated_df['product_category_name_english'].fillna(
    translated_df['product_category_name'].map(missing_map)
)

# 3. Standardize remaining nulls
translated_df['product_category_name_english'] = translated_df['product_category_name_english'].fillna('unknown')

# 4. Finalize table columns
translated_df = translated_df.drop(columns=['product_category_name'])
translated_df = translated_df.rename(columns={'product_category_name_english': 'product_category_name'})



In [67]:
category_groups = {
    # --- HOME AND LIVING ---
    'garden_tools': 'home_n_living',
    'bed_bath_table': 'home_n_living',
    'furniture_decor': 'home_n_living',
    'home_confort': 'home_n_living',
    'housewares': 'home_n_living',
    'construction_tools_lights': 'home_n_living',
    'furniture_living_room': 'home_n_living',
    'furniture_bedroom': 'home_n_living',
    'costruction_tools_garden': 'home_n_living',
    'kitchen_dining_laundry_garden_furniture': 'home_n_living',
    'costruction_tools_tools': 'home_n_living',
    'furniture_mattress_and_upholstery': 'home_n_living',
    'home_construction': 'home_n_living',
    'signaling_and_security': 'home_n_living',
    'christmas_supplies': 'home_n_living',
    'home_appliances_2': 'home_n_living',
    'la_cuisine': 'home_n_living',
    'small_appliances_home_oven_and_coffee': 'home_n_living',
    'home_comfort_2': 'home_n_living',
    'pet_shop': 'home_n_living',
    
    # --- ELECTRONICS & APPLIANCES ---
    'portable_kitchen_and_food_preparers': 'Electronics & Appliances',
    'telephony': 'electronics_n_appliances',
    'fixed_telephony': 'electronics_n_appliances',
    'tablets_printing_image': 'electronics_n_appliances',
    'electronics': 'electronics_n_appliances',
    'computers': 'electronics_n_appliances',
    'computers_accessories': 'Electronics & Appliances',
    'pc_gamer': 'Electronics & Appliances',
    'appliances': 'electronics_n_appliances',
    'small_appliances': 'electronics_n_appliances',
    'audio': 'electronics_n_appliances',
    'air_conditioning': 'electronics_n_appliances',
    'home_appliances': 'electronics_n_appliances',
    'cine_photo': 'electronics_n_appliances',
    'dvds_blu_ray': 'electronics_n_appliances',
    
    


    # --- GROCERIES ---
    'food_drink': 'groceries',
    'food': 'groceries',
    'drinks': 'groceries',

    # --- BABIES & KIDS ---
    'baby': 'babies_n_kids',
    'diapers_and_hygiene': 'babies_n_kids',
    'fashion_childrens_clothes': 'babies_n_kids',


    # --- OFFICE & SCHOOL SUPPLIES ---
    'stationery': 'office_n_school_supplies',
    'office_furniture': 'office_n_school_supplies',
    'books_technical': 'office_n_school_supplies',
    'art': 'office_n_school_supplies',
    'books_general_interest': 'office_n_school_supplies',
    'books_imported': 'office_n_school_supplies',
    'arts_and_craftmanship': 'office_n_school_supplies',

    # --- TOYS, GAMES & HOBBIES ---
    'toys': 'toys_games_n_hobbies',
    'console_games': 'toys_games_n_hobbies',
    'cool_stuff': 'toys_games_n_hobbies',
    'dvds_blu_ray': 'toys_games_n_hobbies',
    'party_supplies': 'toys_games_n_hobbies',
    'flowers': 'toys_games_n_hobbies',
    


    # --- SPORTS & LEISURE ---
    'sports_leisure': 'sports_n_leisure',
    'musical_instruments': 'sports_n_leisure',
    'music': 'sports_n_leisure',
    'cds_dvds_musicals': 'sports_n_leisure',


    # --- FASHION & ACCESSORIES ---
    'watches_gifts': 'fashion_n_accessories',
    'luggage_accessories': 'fashion_n_accessories',
    'fashio_female_clothing': 'fashion_n_accessories',
    'fashion_male_clothing': 'fashion_n_accessories',
    'fashion_bags_accessories': 'fashion_n_accessories',
    'fashion_shoes': 'fashion_n_accessories',
    'fashion_underwear_beach': 'fashion_n_accessories',
    'fashion_sport': 'fashion_n_accessories',
    
    # --- HEALTH & BEAUTY ---
    'health_beauty': 'health_n_beauty',
    'perfumery': 'health_n_beauty',

    # --- INDUSTRIAL TOOLS---
    'auto': 'industrial_tools',
    'agro_industry_and_commerce': 'industrial_tools',
    'industry_commerce_and_business': 'industrial_tools',
    'construction_tools_construction': 'industrial_tools',
    'construction_tools_safety': 'industrial_tools',
    
    # ---OTHERS ---
    'security_and_services': 'others',
    'market_place': 'others',
    'Other': 'others',

    #-- UNKNOWN --
    'unknown': 'unknown'
}
# 1. Map the specific names to the broad groups
translated_df['product_category_group'] = translated_df['product_category_name'].map(category_groups)

# 2. Catch any categories you didn't explicitly group and label them 'Other'
translated_df['product_category_group'] = translated_df['product_category_group'].fillna('Other')
print(translated_df['product_category_group'].value_counts())

count = translated_df['product_category_group'].value_counts()
print(count.sum())

print(len(translated_df))



product_category_group
home_n_living               10633
health_n_beauty              3312
sports_n_leisure             3184
fashion_n_accessories        2894
electronics_n_appliances     2617
industrial_tools             2533
toys_games_n_hobbies         2288
Electronics & Appliances     1652
office_n_school_supplies     1602
babies_n_kids                 936
unknown                       610
Other                         317
groceries                     267
others                        106
Name: count, dtype: int64
32951
32951


In [ ]:
category_groups = {
    # --- HOME AND LIVING ---
    'garden_tools': 'home_n_living',
    'bed_bath_table': 'home_n_living',
    'furniture_decor': 'home_n_living',
    'home_confort': 'home_n_living',
    'housewares': 'home_n_living',
    'construction_tools_lights': 'home_n_living',
    'furniture_living_room': 'home_n_living',
    'furniture_bedroom': 'home_n_living',
    'construction_tools_garden': 'home_n_living', # Fixed Olist spelling ("n" added)
    'kitchen_dining_laundry_garden_furniture': 'home_n_living',
    'construction_tools_tools': 'home_n_living',  # Fixed Olist spelling ("n" added)
    'furniture_mattress_and_upholstery': 'home_n_living',
    'home_construction': 'home_n_living',
    'signaling_and_security': 'home_n_living',
    'christmas_supplies': 'home_n_living',
    'home_appliances_2': 'home_n_living',
    'la_cuisine': 'home_n_living',
    'small_appliances_home_oven_and_coffee': 'home_n_living',
    'home_comfort_2': 'home_n_living',
    'pet_shop': 'home_n_living',
    
    # --- ELECTRONICS & APPLIANCES ---
    'portable_kitchen_and_food_preparers': 'electronics_n_appliances', # Fixed casing
    'telephony': 'electronics_n_appliances',
    'fixed_telephony': 'electronics_n_appliances',
    'tablets_printing_image': 'electronics_n_appliances',
    'electronics': 'electronics_n_appliances',
    'computers': 'electronics_n_appliances',
    'computers_accessories': 'electronics_n_appliances',              # Fixed casing
    'appliances': 'electronics_n_appliances',
    'small_appliances': 'electronics_n_appliances',
    'audio': 'electronics_n_appliances',
    'air_conditioning': 'electronics_n_appliances',
    'home_appliances': 'electronics_n_appliances',
    'cine_photo': 'electronics_n_appliances',
    'dvds_blu_ray': 'electronics_n_appliances',

    # --- GROCERIES ---
    'food_drink': 'groceries',
    'food': 'groceries',
    'drinks': 'groceries',

    # --- BABIES & KIDS ---
    'baby': 'babies_n_kids',
    'diapers_and_hygiene': 'babies_n_kids',
    'fashion_childrens_clothes': 'babies_n_kids',

    # --- OFFICE & SCHOOL SUPPLIES ---
    'stationery': 'office_n_school_supplies',
    'office_furniture': 'office_n_school_supplies',
    'books_technical': 'office_n_school_supplies',
    'art': 'office_n_school_supplies',
    'books_general_interest': 'office_n_school_supplies',
    'books_imported': 'office_n_school_supplies',
    'arts_and_craftmanship': 'office_n_school_supplies',

    # --- TOYS, GAMES & HOBBIES ---
    'toys': 'toys_games_n_hobbies',
    'consoles_games': 'toys_games_n_hobbies', # Fixed Olist naming ("s" added)
    'cool_stuff': 'toys_games_n_hobbies',
    'party_supplies': 'toys_games_n_hobbies',
    'flowers': 'toys_games_n_hobbies',
    'pc_gamer': 'toys_games_n_hobbies',                           # Fixed casing
    
    # --- SPORTS & LEISURE ---
    'sports_leisure': 'sports_n_leisure',
    'musical_instruments': 'sports_n_leisure',
    'music': 'sports_n_leisure',
    'cds_dvds_musicals': 'sports_n_leisure',

    # --- FASHION & ACCESSORIES ---
    'watches_gifts': 'fashion_n_accessories',
    'luggage_accessories': 'fashion_n_accessories',
    'fashion_female_clothing': 'fashion_n_accessories', # Fixed spelling typo
    'fashion_male_clothing': 'fashion_n_accessories',
    'fashion_bags_accessories': 'fashion_n_accessories',
    'fashion_shoes': 'fashion_n_accessories',
    'fashion_underwear_beach': 'fashion_n_accessories',
    'fashion_sport': 'fashion_n_accessories',
    
    # --- HEALTH & BEAUTY ---
    'health_beauty': 'health_n_beauty',
    'perfumery': 'health_n_beauty',

    # --- INDUSTRIAL TOOLS---
    'auto': 'industrial_tools',
    'agro_industry_and_commerce': 'industrial_tools',
    'industry_commerce_and_business': 'industrial_tools',
    'construction_tools_construction': 'industrial_tools',
    'construction_tools_safety': 'industrial_tools',
    
    # --- OTHERS ---
    'security_and_services': 'others_n_misc',
    'market_place': 'others_n_misc',

    # --- UNKNOWN ---
    'unknown': 'unknown_n_missing'
}

# 1. Map the specific names to the broad groups
translated_df['product_category_group'] = translated_df['product_category_name'].map(category_groups)

# 2. Unified fallback to matching snake_case text string
translated_df['product_category_group'] = translated_df['product_category_group'].fillna('others_n_misc')
count = translated_df['product_category_group'].value_counts()
print(count.sum())

print(translated_df['product_category_group'].value_counts())


32951
product_category_group
home_n_living               10506
electronics_n_appliances     4317
health_n_beauty              3312
sports_n_leisure             3184
fashion_n_accessories        2867
toys_games_n_hobbies         2557
industrial_tools             2533
office_n_school_supplies     1602
babies_n_kids                 936
unknown_n_missing             610
groceries                     267
others_n_misc                 260
Name: count, dtype: int64


In [77]:
translated_df['product_name_lenght'] = translated_df['product_category_name'].str.len()
print(translated_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_name_lenght         32951 non-null  int64  
 2   product_description_lenght  32341 non-null  float64
 3   product_photos_qty          32341 non-null  float64
 4   product_weight_g            32949 non-null  float64
 5   product_length_cm           32949 non-null  float64
 6   product_height_cm           32949 non-null  float64
 7   product_width_cm            32949 non-null  float64
 8   product_category_name       32951 non-null  str    
 9   product_category_group      32951 non-null  str    
dtypes: float64(6), int64(1), str(3)
memory usage: 2.5 MB
None


In [ ]:
# 1. Define the exact order of columns you want
# We place the IDs and categories first, followed by dimensions and weights
ordered_columns = [
    'product_id', 
    'product_category_group',          # Your new main category column
    'product_category_name',  # Your English sub-category column
    'product_name_lenght', 
    'product_description_lenght', 
    'product_photos_qty', 
    'product_weight_g', 
    'product_length_cm', 
    'product_height_cm', 
    'product_width_cm'
]

# 2. Reapply the order to your dataframe
translated_df = translated_df[ordered_columns]

# 3. Verify the new layout
print(translated_df.head())


                         product_id    product_category_group  \
0  1e9e8ef04dbcff4541ed26657ea517e5           health_n_beauty   
1  3aa071139cb16b67ca9e5dea641aaa2f  office_n_school_supplies   
2  96bd76ec8810374ed1b65e291975717f          sports_n_leisure   
3  cef67bcfe19066a932b7673e239eb23d             babies_n_kids   
4  9dc1a7de274444849c219cff195d0b71             home_n_living   

  product_category_name  product_name_lenght  product_description_lenght  \
0             perfumery                    9                       287.0   
1                   art                    3                       276.0   
2        sports_leisure                   14                       250.0   
3                  baby                    4                       261.0   
4            housewares                   10                       402.0   

   product_photos_qty  product_weight_g  product_length_cm  product_height_cm  \
0                 1.0             225.0               16.0             

In [79]:
translated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_group      32951 non-null  str    
 2   product_category_name       32951 non-null  str    
 3   product_name_lenght         32951 non-null  int64  
 4   product_description_lenght  32341 non-null  float64
 5   product_photos_qty          32341 non-null  float64
 6   product_weight_g            32949 non-null  float64
 7   product_length_cm           32949 non-null  float64
 8   product_height_cm           32949 non-null  float64
 9   product_width_cm            32949 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 2.5 MB


In [81]:
translated_df = translated_df.astype({
    'product_description_lenght': 'int64',
    'product_photos_qty': 'int64'

})



In [83]:
# 1. Define exactly what data type each column should be
column_types = {
    # These must be whole numbers (integers)
    'product_name_lenght': int,
    'product_description_lenght': int,
    'product_photos_qty': int,
    
    # These must keep their decimals (floats)
    'product_weight_g': float,
    'product_length_cm': float,
    'product_height_cm': float,
    'product_width_cm': float
}

# 2. Loop through the list, fill the nulls with 0, and assign the correct type
for col, dtype in column_types.items():
    translated_df[col] = translated_df[col].fillna(0).astype(dtype)

# 3. Check your final layout and data types

print(translated_df.head())


                         product_id    product_category_group  \
0  1e9e8ef04dbcff4541ed26657ea517e5           health_n_beauty   
1  3aa071139cb16b67ca9e5dea641aaa2f  office_n_school_supplies   
2  96bd76ec8810374ed1b65e291975717f          sports_n_leisure   
3  cef67bcfe19066a932b7673e239eb23d             babies_n_kids   
4  9dc1a7de274444849c219cff195d0b71             home_n_living   

  product_category_name  product_name_lenght  product_description_lenght  \
0             perfumery                    9                         287   
1                   art                    3                         276   
2        sports_leisure                   14                         250   
3                  baby                    4                         261   
4            housewares                   10                         402   

   product_photos_qty  product_weight_g  product_length_cm  product_height_cm  \
0                   1             225.0               16.0             

In [87]:
translated_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_group      32951 non-null  str    
 2   product_category_name       32951 non-null  str    
 3   product_name_lenght         32951 non-null  int64  
 4   product_description_lenght  32951 non-null  int64  
 5   product_photos_qty          32951 non-null  int64  
 6   product_weight_g            32951 non-null  float64
 7   product_length_cm           32951 non-null  float64
 8   product_height_cm           32951 non-null  float64
 9   product_width_cm            32951 non-null  float64
dtypes: float64(4), int64(3), str(3)
memory usage: 2.5 MB


In [92]:
translated_df = translated_df.drop(columns = 'product_description_lenght')

In [93]:
translated_df.head()

,product_id,product_category_group,product_category_name,product_name_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,health_n_beauty,perfumery,9,1,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,office_n_school_supplies,art,3,1,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sports_n_leisure,sports_leisure,14,1,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,babies_n_kids,baby,4,1,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,home_n_living,housewares,10,4,625.0,20.0,17.0,13.0


In [94]:
translated_df.to_csv('../df_cleaned/products_cln.csv')